# GlassBox — watching Gemma 4 think

**Track: AI Shield** · [github.com/AviDhandhania/glassbox](https://github.com/AviDhandhania/glassbox)

Gemma 4 reasons out loud before answering — thousands of tokens nobody reads.
GlassBox makes it reason **seven times**, aligns the traces step by step, and finds
**the exact step** where it started guessing. Then it hands the model back its own
competing readings and lets it repair itself.

### This notebook runs in seconds, with no model download

Every generation and judgement is cached by `(model, seed, temperature, prompt)`.
For a cached question the model is never loaded — `think()` and `p_yes()` return
from `cache.json` before touching the weights. So what follows is **the real
pipeline replaying real recorded outputs**, not a mock: same clustering, same
logit-derived judgements, same thresholds.

If a cell reports `model not found`, the committed cache predates a prompt change
(cache keys include the exact prompt text). Skip to **section 7**, which fetches
the 2.9 GB GGUF and runs everything live.

In [ ]:
!git clone -q https://github.com/AviDhandhania/glassbox.git
%cd glassbox
!ls -la

## 1. The maths, checked offline

No model, no network. Entropy behaviour, clause offsets and verdict thresholds.

In [ ]:
!python glassbox.py test

## 2. A question Gemma 4 gets wrong

> **What is the atomic radius of ununoctium in picometres?**

Gemma answers confidently that ununoctium is **element 111**. It is element 118.

Watch where the seven reasoning traces stop agreeing.

In [ ]:
!python show.py "What is the atomic radius of ununoctium in picometres?"

Most steps are **procedure** — "I will check my knowledge base" — and are not
scored, because only an assertion can be wrong. That filter matters: procedure
steps get reworded freely between samples and scored entropy **1.0**, while the
step carrying the actual error scored **0.59**. Ranking on raw entropy points at
the wrong step entirely.

## 3. A question it knows cold

The same pipeline has to stay quiet when nothing is wrong.

In [ ]:
!python show.py "Who painted the Mona Lisa?"

## 4. Self-repair — the model resolves its own fork

Detection is half a guardrail. We show Gemma its **own** competing readings of the
diverging step, ask which is correct, then rebuild the reasoning prefix with its
choice and re-generate from there.

Nothing external is consulted. The bet is that **recognition beats recall**:
free-recalling an element number is a lookup the model half-remembers, but picking
the right option from a shortlist it produced itself is a judgement it may get right.

In [ ]:
import json, glassbox as g

q = "What is the atomic radius of ununoctium in picometres?"
insp = g.inspect(q)
rep = g.repair(q, insp)

if not rep:
    print("No divergent claim step in the cached run - nothing to adjudicate.")
else:
    print(f"Diverging step {rep['step']}  (entropy {rep['entropy']})\n")
    for i, r in enumerate(rep["readings"]):
        print(f"  {'>>' if i == rep['chosen_index'] else '  '} {r[:100]}")
    print("\nkept its original reading" if rep["was_original"] else "\nCHANGED ITS MIND")
    print(f"\nBEFORE: {rep['answer_before'][:280]}")
    print(f"\nAFTER : {rep['answer_after'][:280]}")

## 5. The judge — why we read logits instead of words

The judge never speaks. We run one forward pass and compare the raw logits of the
`YES` and `NO` tokens:

```python
P(same) = softmax(logit_YES, logit_NO)
```

Asked out loud, a small Gemma has such a heavy `YES` prior that it called
*"Leonardo painted it"* and *"Michelangelo painted it"* the same claim — every
question collapsed to one cluster and every entropy read zero. Six prompt
strategies were measured against 12 labelled pairs: best 10/12, worst 5/12. Asking
"do these *conflict*?" also returned YES to everything.

Reading the logits gives a graded, calibratable probability. It is also *cheaper*
than asking — one prefill, zero tokens generated — and impossible through a hosted
API. **This part of the project exists only because the weights are open.**

In [ ]:
# needs the model (the judge pairs are re-scored live), so only run this
# after the download cell in section 7
# !python glassbox.py judgecheck

## 6. Measured results

Every threshold in the codebase comes from a sweep over labelled data, and each is
**centred in its winning range** rather than parked on the edge — entropy from a
handful of samples moves between runs, and a cut-off sitting a hundredth from real
data flips on noise.

In [ ]:
import json, pathlib

for name, title in [("eval_results.json", "Answer-level detection"),
                    ("trace_results.json", "Step-level localization"),
                    ("repair_results.json", "Self-repair"),
                    ("ablation_results.json", "Thinking mode on/off")]:
    p = pathlib.Path(name)
    print(f"=== {title} ".ljust(60, "="))
    if not p.exists():
        print("  not yet run\n")
        continue
    d = json.loads(p.read_text(encoding="utf-8"))
    print(json.dumps(d.get("summary", {"rows": len(d.get('rows', []))}), indent=2), "\n")

## 7. Running a fresh question

Everything above replayed cached runs. To ask something new, fetch the weights —
a single 2.9 GB GGUF that serves as **both** the reasoner and its own judge.

A cold question costs roughly seven reasoning traces plus the judgements, which is
several minutes on CPU. There is no server, no API key, and no network once the
file is on disk.

In [ ]:
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu --only-binary :all:
!mkdir -p models
!curl -L -C - -o models/gemma-4-E2B-it-Q4_K_M.gguf \
  https://huggingface.co/unsloth/gemma-4-E2B-it-GGUF/resolve/main/gemma-4-E2B-it-Q4_K_M.gguf

In [ ]:
!python glassbox.py judgecheck

In [ ]:
!python show.py "In which year did Ronald Fisher publish his first paper on the Behrens-Fisher problem?"

## What this does not catch

**Consistent false belief.** If all seven traces make the *same* wrong move,
entropy is zero and we call it stable. GlassBox measures whether a model is
**inventing on the spot**, not whether it is **right**. Confabulation is unstable
and shows up as spread; a memorised falsehood is stable and does not. Catching that
needs retrieval against a source — a different tool, honestly labelled.

**A finding worth its own line:** Gemma 3 confabulated freely on false premises.
Gemma 4 with thinking mode refuses or corrects nearly all of them — including
*"Which Apollo mission first landed on the far side of the Moon?"*, which Gemma 3
answered "Apollo 17". Thinking mode is doing real safety work. What survives it is
subtler: **partial knowledge about real entities**, where the model half-remembers
and fills the gap. That is what GlassBox targets.